# Immediate Interactive Shutdown

**Design epic:** `bd-28i8`  
**Status:** Approved for implementation on 2026-09-04  
**Scope:** The interactive process-exit path reached after Ctrl+C / confirmed quit.

## Goal

Make confirmed interactive shutdown feel immediate by eliminating stacked graceful-drain windows from the process-exit path, while still restoring the terminal, persisting the draft, and terminating child processes.

## Accepted policy

A confirmed process exit may discard in-flight agent/tool output. Final cost-ledger and telemetry writes are best effort. Normal brain retirement and warm session replacement remain graceful and unchanged.

## Current behavior and measured bounds

The Ctrl+C path is not telemetry shutdown. The TUI first saves the draft in `App::confirm_quit`, then CLI teardown calls `InteractiveFrontendHost::shutdown`.

The current shutdown path stacks independent grace windows:

- The frontend host waits 2 seconds for orchestrator completion, then drops channels and can wait another 30 seconds.
- Brain retirement reaches `shutdown_mcp_server`, which performs two sequential 5-second waits: callback-server shutdown and its guard.
- Worker MCP and project-loop runtime shutdown each permit a further 5-second drain.
- Agent transports have their own 1–3 second shutdown bounds.

That composition explains the observed 5–10 second stalls. The latency is policy-induced, not attributable to `spur_telemetry::shutdown`.

## Shutdown contract

Once quit is confirmed:

1. **Fence ingress:** cancel global work and prevent new brain/worker/tool work from being admitted.
2. **Launch aborts concurrently:** drop the active brain transport, abort the worker tree, abort the root MCP task tree, and stop the project runtime.
3. **Join one bounded barrier:** wait only for abort/drop unwinding and child-process ownership release, under the existing host-level bound.
4. **Exit:** never return from shutdown before the resource-stop barrier has completed or the host-level emergency bound has fired.

There are no per-subsystem graceful drain windows on this fast path. Graceful `retire_active_brain` behavior remains the session swap/restart path.

In [ ]:
stateDiagram-v2
    [*] --> Running
    Running --> Fenced: confirm_quit
    Fenced --> Aborting: launch_parallel_abort
    Aborting --> ResourcesStopped: all_children_joined
    ResourcesStopped --> Exited: exit

    note right of Running
      @spec IMMEDIATE-SHUTDOWN
      @type ShutdownEvent = enum[confirm_quit, launch_parallel_abort, all_children_joined, exit]
      @input event: ShutdownEvent
      @state-var ingress_fenced: Bool
      @state-var aborts_issued: Bool
      @state-var resources_stopped: Bool
      @state-var process_exited: Bool
      @requires PRE: ingress_fenced = false and aborts_issued = false and resources_stopped = false and process_exited = false
      @state Running
      @invariant SAFE_EXIT: not process_exited or resources_stopped
    end note

    note right of Fenced
      @state Fenced
      @transition QUIT
      @from Running
      @to Fenced
      @event event = confirm_quit
      @guard ingress_fenced = false
      @update ingress_fenced' = true
      @update aborts_issued' = aborts_issued
      @update resources_stopped' = resources_stopped
      @update process_exited' = process_exited
    end note

    note right of Aborting
      @state Aborting
      @transition ABORT
      @from Fenced
      @to Aborting
      @event event = launch_parallel_abort
      @guard ingress_fenced = true and aborts_issued = false
      @update ingress_fenced' = ingress_fenced
      @update aborts_issued' = true
      @update resources_stopped' = resources_stopped
      @update process_exited' = process_exited
    end note

    note right of ResourcesStopped
      @state ResourcesStopped
      @transition JOIN
      @from Aborting
      @to ResourcesStopped
      @event event = all_children_joined
      @guard aborts_issued = true and resources_stopped = false
      @update ingress_fenced' = ingress_fenced
      @update aborts_issued' = aborts_issued
      @update resources_stopped' = true
      @update process_exited' = process_exited
    end note

    note right of Exited
      @state Exited
      @transition EXIT
      @from ResourcesStopped
      @to Exited
      @event event = exit
      @guard resources_stopped = true and process_exited = false
      @update ingress_fenced' = ingress_fenced
      @update aborts_issued' = aborts_issued
      @update resources_stopped' = resources_stopped
      @update process_exited' = true
      @verify INIT_SAFE_EXIT: prove initiate SAFE_EXIT
      @verify QUIT_SAFE_EXIT: prove preserve SAFE_EXIT on QUIT
      @verify ABORT_SAFE_EXIT: prove preserve SAFE_EXIT on ABORT
      @verify JOIN_SAFE_EXIT: prove preserve SAFE_EXIT on JOIN
      @verify EXIT_SAFE_EXIT: prove preserve SAFE_EXIT on EXIT
    end note

## Implementation seams

- Add an explicit immediate-exit orchestration path in `spur-core`; do not weaken the existing graceful retirement contract.
- Make consuming/dropping the active `AgentConnection` an intentional fast-stop operation so adapter `Drop` implementations and Tokio `kill_on_drop` ownership terminate children immediately.
- Reuse `McpCallbackServer::force_abort_and_wait` for the root MCP task tree.
- Add a worker-MCP fast-stop operation that cancels ingress and aborts handler tasks without a grace delay.
- Add a project-runtime fast-stop operation for process exit; supervisor restart/retirement behavior remains graceful.
- Have `InteractiveFrontendHost::shutdown` invoke the fast path and use one outer emergency bound, removing the second 30-second wait.

The independent resource stops should be joined concurrently after the fence. Sequential subsystem timeouts are specifically forbidden on the process-exit path.

## Verification and decision evidence

Each implementation task follows:

1. Load the relevant versioned solver rule-family summary.
2. Run and persist a task-specific pre-solve.
3. Add a targeted regression test and observe the expected failure.
4. Implement the smallest behavior change and observe the test pass.
5. Run and persist a post-solve against the landed lifecycle.
6. Run crate-level formatting, tests, and lint checks before completion.

Design solves:

- Parallel fast-path schedule: `sol_f7a08872f74f46e6` (4 abstract lifecycle ticks; a strict sequential chain required 6).
- Safety workflow: `sol_0de65009df784b67` (safe path reaches exit only after resources stop; an early-exit variant produced a counterexample).

## Risks and non-goals

- In-flight responses, notifications, cost updates, and telemetry flushing may be lost after confirmed exit.
- Terminal restoration and draft persistence are not relaxed.
- Child-process termination is mandatory; dropping task handles without dropping their owned transports is insufficient.
- This design does not change graceful brain swaps, restarts, or ordinary session retirement.